In [1]:
import numpy as np
import datetime
import Bayesian_Methods as Bayes
from functools import partial
import sys
import Parameters as Pars
from Full_Par_Method import Parameter_Processing
import Full_Par_Method
from multiprocess import Pool
from dynesty import DynamicNestedSampler, sampler, plotting
from getdist import plots, MCSamples

In [4]:
Data = Parameter_Processing(Full_Par_Method.FidPars, load_strain = False)
Nano_strains = Data.CharStrainBound_Nano_15()[0]
ndim = 6
par_ranges = ((-3.5,10.5,7.5,0,0.1,-1.5),(-1.5,12.5,9.5,1,11,0))
par_names = (Pars.psi0[0], Pars.m0[0], Pars.mass_norm[0], Pars.scatter[0], Pars.tau[0], Pars.gamma_inner[0])

In [26]:
parameters = list(Full_Par_Method.FidPars.values())
parameters[0] = -1.5
parameters[1] = 12.5

In [28]:
timeholo1 = datetime.datetime.now()
par_names = (Pars.psi0[0], Pars.m0[0], Pars.mass_norm[0], Pars.scatter[0], Pars.tau[0], Pars.gamma_inner[0])
par_dict = {key: value for key, value in zip(par_names,parameters)}
t1 = Parameter_Processing(par_dict, load_strain = False)
t1.get_Strain_pdf(provide_kde = Nano_strains)
timeholo2 = datetime.datetime.now()
timelike1 = datetime.datetime.now()
DistObj = Bayes.Distribution(par_dict) #Initialize Distribution Object for the pre-defined parameters
PDF = DistObj.PDF(DistObj) #Create a pdf object from the given distribution object
PDF.get_pdf(t1.pdf, t1.hcmax)
PDF.dim_names = np.array('Realisations') #Set the name of the dimensions of the PDF to the dimensions of holodeck
Priors = DistObj.Priors(DistObj)
Priors.Uniform()
I = DistObj.MCI(Priors,PDF)  
I.apply_prior()
Likelihood = I.Integration('Realisations')
print(np.log(Likelihood.flatten()[0]))
timelike2 = datetime.datetime.now()

print(f'The time taken for holodeck to run the simulations is {timeholo2-timeholo1} seconds.')
print(f'The time taken to calculate the likelihood is {timelike2 - timelike1} seconds.')

12:18:06 INFO : No galaxy pair-fraction given, using galaxy merger-rate. [sam.py:__init__]
12:18:07 INFO : No GMT was provided, cannot calculate Galaxy-Merger based stalling. [sam.py:static_binary_density]
12:18:07 INFO : Adding MMbulge scatter (4.5000e-01) [sam.py:static_binary_density]
12:18:07 INFO : 	dens bef: (2.83e-104, 9.28e-51, 7.10e-06, 5.56e-01, 2.90e+01, 1.21e+02, 9.34e+02) [sam.py:static_binary_density]
12:18:24 INFO : Scatter added after 17.090978 sec [sam.py:static_binary_density]
12:18:25 INFO : 	dens aft: (1.84e-11, 7.25e-05, 1.38e-02, 2.31e+00, 5.59e+01, 1.42e+02, 5.38e+02) [sam.py:static_binary_density]
12:18:25 INFO : 	mass: 2.20e+01 ==> 5.29e+01 || change = 1.4061e+00 [sam.py:static_binary_density]
12:18:25 ERROR : Warning, significant change in number-mass!  mass: 2.20e+01 ==> 5.29e+01 || change = 1.4061e+00 [sam.py:static_binary_density]
12:18:25 INFO : `gmt_time` not calculated in SAM.  Setting to zeros. [sam.py:gwb]
-87.31362352281863
The time taken for holodeck